# 2 — Démonstration temps réel (interface lisible)

Version avec texte plus lisible (fonds sombres, police plus grande) et **correction des lettres répétées** (délai minimum entre deux lettres validées).

**Prérequis :** avoir exécuté le notebook 1 (`asl_mediapipe_mlp_model.h5` + `labels.json`).

### Contrôles
- **Espace** → espace · **`d`** ou **Retour arrière** → effacer · **`c`** → tout effacer · **`p`** → voix (option) · **`q`** → quitter


In [2]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import json, time
from collections import deque

model = tf.keras.models.load_model("asl_mediapipe_mlp_model.h5")
labels = json.load(open("labels.json"))
print("Classes :", labels)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7, max_num_hands=1)

TTS_OK = False
try:
    import pyttsx3
    _tts = pyttsx3.init()
    TTS_OK = True
except Exception:
    TTS_OK = False

def speak(text):
    if TTS_OK and text.strip():
        try:
            _tts.say(text); _tts.runAndWait()
        except Exception:
            pass

print("Synthese vocale :", "disponible (touche p)" if TTS_OK else "indisponible (pip install pyttsx3)")


2026-07-07 12:42:57.029069: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-07 12:42:57.030074: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-07 12:42:57.045655: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-07 12:42:57.045670: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-07 12:42:57.046149: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Classes : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'space']
Synthese vocale : indisponible (pip install pyttsx3)


I0000 00:00:1783420978.159896   99275 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1783420978.163476   99444 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) Graphics (ARL)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [ ]:
def normalize_landmarks(landmarks):
    lm = landmarks.astype(np.float32).copy()
    lm -= lm[0]
    scale = np.linalg.norm(lm[9, :2])
    if scale < 1e-6:
        scale = 1e-6
    lm /= scale
    return lm.flatten().reshape(1, -1)

def draw_bar(frame, x, y, w, h, frac, color):
    frac = max(0.0, min(1.0, frac))
    cv2.rectangle(frame, (x, y), (x + w, y + h), (60, 60, 60), -1)
    cv2.rectangle(frame, (x, y), (x + int(w * frac), y + h), color, -1)

FONT = cv2.FONT_HERSHEY_SIMPLEX

# texte lisible : fond sombre + lissage
def put_label(frame, text, org, scale=0.7, color=(255, 255, 255), thick=2, bg=(0, 0, 0)):
    (tw, th), base = cv2.getTextSize(text, FONT, scale, thick)
    x, y = org
    cv2.rectangle(frame, (x - 5, y - th - 7), (x + tw + 5, y + base + 3), bg, -1)
    cv2.putText(frame, text, (x, y), FONT, scale, color, thick, cv2.LINE_AA)

CONF_THRESHOLD = 0.80
STAB_WINDOW = 5
STAB_THRESHOLD = 4
COOLDOWN = 0.6          # delai minimum (s) entre deux lettres validees -> stoppe les repetitions
window = deque(maxlen=STAB_WINDOW)
sentence = ""
committed = None
last_commit_t = 0.0
prev_t = 0.0

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
win = "Reconnaissance ASL - Demo"
cv2.namedWindow(win, cv2.WINDOW_NORMAL)
cv2.resizeWindow(win, 1280, 720)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    top_label = "nothing"
    pred = None
    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark])
        pred = model.predict(normalize_landmarks(coords), verbose=0)[0]
        idx = int(np.argmax(pred)); conf = float(pred[idx])
        if conf >= CONF_THRESHOLD:
            top_label = labels[idx]

    # stabilisation + delai anti-repetition
    window.append(top_label)
    stable = top_label if window.count(top_label) >= STAB_THRESHOLD else None
    if stable is not None and stable != committed:
        if stable == "nothing":
            committed = None                                  # main retiree -> prete pour la suivante
        elif (time.time() - last_commit_t) >= COOLDOWN:
            if stable == "space":
                sentence += " "
            elif stable == "del":
                sentence = sentence[:-1]
            else:
                sentence += stable
            committed = stable
            last_commit_t = time.time()

    # ===== HUD lisible =====
    now = time.time()
    fps = 1.0 / (now - prev_t) if prev_t else 0.0
    prev_t = now
    put_label(frame, "espace=espace   d=effacer   c=tout   p=voix   q=quitter", (15, 32), 0.6)
    put_label(frame, f"FPS {fps:2.0f}", (15, 66), 0.6)

    if pred is not None:
        top3 = np.argsort(pred)[::-1][:3]
        for i, ci in enumerate(top3):
            yb = 100 + i * 34
            put_label(frame, str(labels[ci]), (15, yb + 20), 0.7)
            draw_bar(frame, 70, yb, 200, 24, float(pred[ci]), (0, 200, 0))
            put_label(frame, f"{pred[ci]*100:3.0f}%", (285, yb + 20), 0.6)

    prog = window.count(top_label) / STAB_THRESHOLD
    draw_bar(frame, 70, 210, 200, 18, prog, (0, 150, 255))
    put_label(frame, "stabilite", (285, 226), 0.6)

    h, w, _ = frame.shape
    cv2.rectangle(frame, (0, h - 66), (w, h), (0, 0, 0), -1)
    put_label(frame, sentence[-40:], (20, h - 22), 1.1, (255, 255, 255), 2, (0, 0, 0))

    cv2.imshow(win, frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key in (ord('d'), 8):
        sentence = sentence[:-1]
    elif key == 32:
        sentence += " "
    elif key == ord('c'):
        sentence = ""
    elif key == ord('p'):
        speak(sentence)

cap.release()
cv2.destroyAllWindows()
